# CS 3892 / 5892 — Session 11 · Linear temporal logic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-10-01-linear-temporal-logic.ipynb)

**Thursday, October 1, 2026.** An invariant says what is true in every *state*. LTL says
what is true along a *run* — an infinite sequence of states — using four operators:
`X` (next), `G` (globally), `F` (finally), `U` (until).

Everything here runs **NuSMV** (free, LGPL), fetched from FBK by the setup cells, on the
model files in `sessions/cs3892-2026-10-01-linear-temporal-logic/`. Every verdict printed below is also **asserted**, so
if a cell finishes, the result is the one on the slide.

The same files open in the browser in **smvis** — [bit.ly/fmaiv_smvis](https://bit.ly/fmaiv_smvis) —
see the last section.

## Setup

Run these two once.

In [ ]:
# --- Setup: find the repo (clone on Colab) and define helpers ---------------
import os, re, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-10-01-linear-temporal-logic"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

SMV = ROOT / "sessions" / SESSION / "smv"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path):
    """Run one Python example and stream its output. Raises if it fails."""
    path = pathlib.Path(path)
    show(path)
    r = subprocess.run([sys.executable, str(path)], capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.returncode != 0:
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

In [ ]:
# --- NuSMV: fetch the official FBK build if it is not already installed ------
# NuSMV is free (LGPL) from nusmv.fbk.eu. (Same cell as sessions 9 and 10.)
# 2.7.1 is tried first; it needs glibc >= 2.38 and libedit, which an older
# Colab image may lack, so the static 2.6.0 build is the fallback. Same verdicts.
import shutil

BUILDS = [("2.7.1", "https://nusmv.fbk.eu/distrib/2.7.1/NuSMV-2.7.1-linux64.tar.xz", "J"),
          ("2.6.0", "https://nusmv.fbk.eu/distrib/NuSMV-2.6.0-linux64.tar.gz",       "z")]

def _works(exe):
    """True only if this binary actually model-checks a file -- a missing
    library makes it exit 127 with an error that still contains "NuSMV"."""
    try:
        r = subprocess.run([exe, str(SMV / "01_counter_operators.smv")],
                           capture_output=True, text=True, timeout=60)
        return "-- specification" in r.stdout
    except Exception:
        return False

NUSMV = shutil.which("NuSMV")
if not (NUSMV and _works(NUSMV)):
    NUSMV = None
    if os.geteuid() == 0 and shutil.which("apt-get"):      # Colab runs as root
        subprocess.run("apt-get install -y -qq libedit2 >/dev/null 2>&1", shell=True)
    for ver, url, z in BUILDS:
        dest = pathlib.Path.home() / ".local" / f"nusmv-{ver}"
        dest.mkdir(parents=True, exist_ok=True)
        subprocess.run(f"curl -sSL {url} | tar -x{z} -C {dest} --strip-components=1",
                       shell=True, check=True)
        if _works(str(dest / "bin" / "NuSMV")):
            NUSMV = str(dest / "bin" / "NuSMV")
            break
        print(f"NuSMV {ver} will not start on this machine -- trying the next build")
assert NUSMV, "could not install NuSMV"
print("NuSMV:", NUSMV)

def smv(path):
    """Run NuSMV on a file in batch mode; return its output minus the banner."""
    r = subprocess.run([NUSMV, str(path)], capture_output=True, text=True)
    return "\n".join(l for l in (r.stdout + r.stderr).splitlines() if not l.startswith("***"))

def results(out):
    """Split NuSMV output into {printed formula: (verdict, counterexample text)}.
    NuSMV does not print verdicts in file order, so match them by formula."""
    res, key = {}, None
    for line in out.splitlines():
        m = re.match(r"-- (?:specification|invariant)\s+(.*?)\s+is (true|false)$", line)
        if m:
            key = re.sub(r"\s+", " ", m.group(1)).strip()
            res[key] = [m.group(2), ""]
        elif key and res[key][0] == "false":
            res[key][1] += line + "\n"
    return {k: tuple(v) for k, v in res.items()}

def verdicts(path):
    """Print one line per property, and return {formula: verdict}."""
    out = smv(path)
    res = results(out)
    for k, (v, _) in res.items():
        print(f"{v:6} {k}")
    return {k: v for k, (v, _) in res.items()}

print("ready -- smv(path), results(out), verdicts(path)")

## 1. The four operators, on the running example

Slides 8–11. The counter from `labs/running-example.md`: it has exactly **one** run,
`0, 1, …, 10, 0, 1, …, 10, 0, …`, and every formula is judged at its first state.

| formula | reads | verdict |
|---|---|---|
| `X (x = 1)` | the next state has `x = 1` | true |
| `G (x <= 10)` | always — safety | true |
| `F (x = 10)` | eventually | true |
| `(x < 10) U (x = 10)` | `x < 10` until `x = 10` | true |
| `(x < 5) U (x = 10)` | `x < 5` until `x = 10` | **false** — `x = 5` comes first |
| `G (x = 10 -> X (x = 0))` | every cap is followed by a reset | true |
| `G F (x = 0)` | infinitely often | true |
| `F G (x = 10)` | from some point on, forever | **false** — a lasso |

In [ ]:
F = SMV / "01_counter_operators.smv"
show(F)
v = verdicts(F)
assert list(v.values()) == ["true", "true", "true", "true", "false", "true", "true", "false"], v
print("\nverdicts as on the slides")

## 2. Safety versus liveness

Slides 14–18. Two processes share a lock; a scheduler runs one of them per step, and the
model checker plays the scheduler.

- **Mutual exclusion** `G !(crit_a & crit_b)` — safety — **true**.
- `!crit_a` checked twice, as an **invariant** and as the LTL formula `G !crit_a`. Both
  false. Compare the counterexamples: the invariant's **stops at the bad state**; the LTL
  one is printed as a **lasso**, because every LTL counterexample is an infinite run — but
  for a safety property the loop is decoration. The prefix up to the bad state is the evidence.
- **No starvation** `G (a = trying -> F crit_a)` — liveness — **false**. No finite prefix
  could refute it; the counterexample is a lasso in which `a` waits forever while `b`
  keeps going round. **Here the loop is the evidence.**

In [ ]:
F = SMV / "02_mutex_safety_liveness.smv"
show(F)
res = results(smv(F))
for k, (v, _) in res.items():
    print(f"{v:6} {k}")

assert res["G !(crit_a & crit_b)"][0] == "true"
inv, ltl, live = res["!crit_a"], res["G !crit_a"], res["G (a = trying -> F crit_a)"]
assert inv[0] == ltl[0] == live[0] == "false"

print("\n--- the invariant's counterexample: it stops at the bad state")
print(inv[1])
print("--- the same statement in LTL: an infinite run, printed as a lasso")
print(ltl[1])
print("--- liveness: a waits forever while b cycles")
print(live[1])
assert "Loop starts here" not in inv[1]
assert "Loop starts here" in ltl[1] and "Loop starts here" in live[1]
print("as on the slides")

## 3. English to LTL — two formulas that are true, and wrong

Slides 21–22. *"The agent never deletes anything a human has not approved."* This copy of
the agent has a bug: an "urgent" shortcut from `plan` straight to the delete.

| formula | what it actually says | on the buggy agent |
|---|---|---|
| `G (del -> approved)` | the requirement | **false** — catches the shortcut |
| `G del -> approved` | `(G del) -> approved`: *if* it deletes at every step, … | **true** — vacuously |
| `G (approved -> F del)` | every approval is followed by a delete | **true** — the converse |

Look at how NuSMV echoes (b): `( G del -> approved)`. It wraps the *outermost* operator in
parentheses, so the arrow is outermost — `(G del) -> approved`. smvis prints it as
`(G (del) -> approved)`.

In [ ]:
F = SMV / "03_agent_approval_buggy.smv"
show(F)
v = verdicts(F)
assert v == {"G (del -> approved)": "false",
             "( G del -> approved)": "true",
             "G (approved -> F del)": "true"}, v
res = results(smv(F))
print("\n--- the counterexample: plan, then a delete no one approved")
print(res["G (del -> approved)"][1])

## 4. Strong versus weak *until*

Slide 23. The **fixed** agent: no shortcut. `p U q` insists that `q` **eventually happens**;
the requirement never did. `p U q` is safety and liveness at once —
`p U q  =  (p W q) & F q` — and NuSMV has no `W`, so weak until is written `(p U q) | G p`.

In [ ]:
F = SMV / "04_agent_approval_fixed.smv"
show(F)
v = verdicts(F)
assert v == {"G (del -> approved)": "true",
             "(!del U approved)": "false",
             "((!del U approved) | G !del)": "true"}, v
res = results(smv(F))
print("\n--- why the strong until fails: every task abandoned, forever. Nothing deleted.")
print(res["(!del U approved)"][1])

## 5. Temporal properties over agent logs

Slide 25. A deployed agent gives you a **finite log**, not a model. A runtime monitor reads
it with the same formulas and can only ever answer **violated**, **satisfied** or
**inconclusive** — and which of those a property can ever return depends on whether it is
safety or liveness.

In [ ]:
run(PYD / "05_monitor_agent_logs.py")

## 6. The same files in smvis

Slides 26–27.

[bit.ly/fmaiv_smvis](https://bit.ly/fmaiv_smvis) runs NuSMV in the browser and draws the
state graph. Nothing to install.

1. Open a file above (`show(...)` prints it) and paste it into the **SMV Editor**, or use
   **Upload .smv**.
2. **Parse**, then **Compute All** — the reachable state graph appears.
3. **Check All Specs** — one ✓ or ✗ per property. (smvis labels a passing LTL property
   `SPEC`; that is a display quirk, not a different check.)
4. Pick a failing property in the trace box: its counterexample is drawn on the graph.
5. Under **LTL Buchi Analysis**, pick an `LTLSPEC` and **Compose & Analyze**: smvis negates
   the formula, builds a Büchi automaton for the negation, and searches the product for an
   accepting cycle — the lasso. That is **Tuesday's lecture**, running.

`03_agent_approval_buggy.smv` has the smallest graph (six reachable states) and is the one
to try first.

## Try it yourself

- In `02`, add `LTLSPEC G (b = trying -> F crit_b)`. Predict the verdict before you run it.
- In `02`, is `G F crit_a` true? What does the scheduler have to do to falsify it?
- In `04`, write *"the agent eventually deletes something"* and *"the agent deletes
  infinitely often"*. Which are true, and what run refutes the false one?